# 🔴 Blast Radius Detector — v5.0
**Entrega Final | Machine Learning II / Ciência de Dados**

| Campo | Detalhe |
|---|---|
| **Versão** | 5.0 — Final Production-Ready |
| **Arquitetura** | 3 camadas com votação ponderada + feedback loop persistente |
| **Modelo NLP** | XLM-RoBERTa-Large-XNLI (multilingual) |
| **Dataset** | 502 exemplos balanceados (discrepância da v4 corrigida) |
| **Meta F1** | ≥ 0,75 weighted (CV 5-fold) |

---

## O que é Blast Radius?

O **Raio de Impacto** (*Blast Radius*) categoriza a escala do dano potencial de um comando antes de sua execução, separando operações inofensivas de ações catastróficas.

| Nível | Exemplo | Impacto |
|---|---|---|
| 🟢 **Baixo Risco** | `cat file.txt` | Restrito ao ambiente local |
| 🟡 **Médio Risco** | `DELETE FROM logs WHERE date < '2023-01-01'` | Afeta estruturas específicas |
| 🟠 **Alto Risco** | `DROP TABLE users` | Destrói estruturas de dados |
| 🔴 **Crítico** | `DROP DATABASE production` | Falha catastrófica sistêmica |

---

## Evolução por Versão

| Melhoria | v1.0 | v2.0 | v3.0 | v4.0 | v5.0 |
|---|---|---|---|---|---|
| Tamanho do dataset | 15 | 54 | 180 | 344 ¹ | **502** ✅ |
| F1 weighted (CV 5-fold) | — | 0,479 ± 0,138 | 0,671 ± 0,052 | 0,689 ± 0,032 | **calculado abaixo** (meta ≥ 0,75) |
| Modelo NLP | — | BART-MNLI | XLM-RoBERTa | XLM-RoBERTa + threshold | XLM-RoBERTa calibrado |
| Divergência | Ignorada | Alerta | 55,6% | 30,8% | **Reduzida** |
| Feedback loop | ❌ | ❌ | ❌ | ✅ Lookup exato | **✅ Re-treino incremental** |
| Consistência dataset | ❌ | ❌ | ❌ | ❌ discrepância | **✅ Assertion verificada** |

> ¹ **Nota sobre a v4:** O Markdown anunciava ~500 exemplos, mas o `classification_report`
> registrava `support = 344`. A v5 corrige essa inconsistência com uma `assert` explícita que
> interrompe a execução caso os números não batam, garantindo total de **502 exemplos reais**.

---

## Objetivos da v5.0

1. **Corrigir a discrepância do dataset** — alinhar o total anunciado com o real (502 exemplos verificados)
2. **Atingir meta F1 ≥ 0,75** — não alcançada na v4 (0,689); abordada via expansão criteriosa
3. **Feedback loop com re-treino incremental** — exportação CSV e reinjeção no dataset
4. **Curva de aprendizado** — diagnóstico explícito de overfitting (gap treino/validação)
5. **Auditabilidade** — cada predição registra votos e confiança das 3 camadas


## ⚙️ 1. Configuração do Ambiente

Instalamos as dependências do motor de inteligência:
- **`transformers`** — Modelo XLM-RoBERTa-XNLI (Hugging Face, multilingual)
- **`scikit-learn`** — Random Forest com vetorização TF-IDF
- **`networkx`** — Grafo dinâmico de propagação de impacto
- **`ipywidgets`** — Console interativo de análise


In [ ]:
!pip install transformers scikit-learn networkx ipywidgets -q

## 📦 2. Importação de Bibliotecas e Inicialização do Modelo

### Por que XLM-RoBERTa-Large-XNLI?

| Critério | BART-MNLI (v1/v2) | XLM-RoBERTa-XNLI (v3→v5) |
|---|---|---|
| Idiomas | Inglês (primário) | 15 idiomas |
| `ls -la` falso positivo | ❌ Alto Risco | ✅ Baixo Risco |
| Dados de inferência | Menos diversificados | XNLI multi-idioma |

### Threshold de Confiança NLP

Na v3, o NLP retornava `crítico` para `ls -la` com ~41% de confiança. A partir da v4,
implementamos um **threshold mínimo de 50%**: abaixo disso, o NLP abstém-se e cede ao
voto da heurística. A v5 mantém esse mecanismo e documenta a taxa de abstenção na bateria
de testes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import ipywidgets as widgets
import warnings
import csv
import datetime
from io import StringIO
import scipy.sparse as sp
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (cross_val_score, StratifiedKFold,
                                     learning_curve)
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from transformers import pipeline
from IPython.display import display, HTML, clear_output
from collections import Counter

# ── Inicialização do Modelo NLP ───────────────────────────────────────────────
print('⏳ Carregando modelo XLM-RoBERTa-XNLI (Hugging Face)...')
classifier_hf = pipeline(
    'zero-shot-classification',
    model='joeddav/xlm-roberta-large-xnli'
)
labels_hf = ['low risk', 'medium risk', 'high risk', 'critical']

mapa_labels_display = {
    'low risk':    'baixo risco',
    'medium risk': 'médio risco',
    'high risk':   'alto risco',
    'critical':    'crítico',
}

NLP_CONFIDENCE_THRESHOLD = 0.50   # Abaixo disso o NLP abstém-se

print('✅ Modelo XLM-RoBERTa-XNLI carregado com sucesso!')
print(f'   Threshold de confiança NLP: {NLP_CONFIDENCE_THRESHOLD*100:.0f}%')


## 🧠 3. Dataset Expandido — 502 Exemplos (Correção da Discrepância v4)

### Problema identificado na v4

O Markdown da v4 anunciava `~500 exemplos`, mas o `classification_report` registrava
`support = 344`. Essa inconsistência comprometia a credibilidade dos resultados.

### Estratégia de expansão na v5

1. **Contagem explícita e verificada** — cada lista tem seu tamanho auditável antes da
   concatenação, e uma `assert` interrompe a execução se o total divergir.
2. **Variações sintáticas reais** — mesmos comandos com parâmetros e paths diferentes.
3. **Cobertura cloud-native** — CI/CD, operações de segurança, AWS RDS, GCloud SQL, Helm.
4. **Edge cases** — comandos ambíguos que causavam divergência em versões anteriores.

| Versão | Total | Baixo Risco | Médio Risco | Alto Risco | Crítico |
|---|---|---|---|---|---|
| v3.0 | 180 | 63 | 44 | 40 | 33 |
| v4.0 | **344** (anunciado ~500) | 155 | 77 | 58 | 54 |
| **v5.0** | **502 ✅** | **155** | **123** | **118** | **106** |


In [ ]:
# ── Dataset v5.0 ──────────────────────────────────────────────────────────────
# As listas são definidas separadamente para permitir auditoria por categoria.
# Uma assertion ao final garante que o total real bate com o documentado.

baixo_risco = [
    # Leitura / navegação
    'ls -la', 'ls -la /home', 'ls /var/www', 'ls /opt/app', 'ls -lh /tmp',
    'cd /home', 'cd /opt/app', 'cd /var/log', 'cd /etc', 'cd /usr/local/bin',
    'pwd', 'pwd && ls', 'cat file.txt', 'cat /etc/hostname', 'cat /etc/os-release',
    'cat requirements.txt', 'cat package.json', 'cat Makefile', 'cat .env.example',
    'cat /etc/hosts', 'cat /proc/cpuinfo', 'cat /proc/meminfo',
    'head -n 10 data.csv', 'head -n 5 app.log', 'head -c 100 file.bin',
    'tail -f app.log', 'tail -n 50 error.log', 'tail -f /var/log/syslog',
    'less /var/log/auth.log', 'more /etc/fstab',
    # Informação do sistema
    'echo hello world', 'echo $PATH', 'echo $HOME', 'echo $USER',
    'whoami', 'id', 'date', 'date +%Y-%m-%d', 'uptime', 'hostname',
    'uname -a', 'uname -r', 'uname -m',
    'df -h', 'df -i', 'free -m', 'free -h',
    'ps aux', 'ps -ef', 'ps aux | grep python',
    'top -bn1', 'htop', 'vmstat', 'iostat',
    'lsblk', 'blkid', 'mount', 'findmnt',
    'env', 'printenv HOME', 'printenv PATH',
    'ifconfig', 'ip addr show', 'ip route show', 'ip link show',
    'netstat -an', 'ss -tulnp', 'ss -s',
    'which python', 'which bash', 'which node',
    'type bash', 'man ls', 'help echo',
    'history | tail -20', 'alias ll="ls -la"',
    # Desenvolvimento
    'python script.py', 'python -c "print(1+1)"', 'python --version',
    'python3 --version', 'python3 manage.py check',
    'node -e "console.log(42)"', 'node --version',
    'java -version', 'gcc --version', 'go version', 'ruby --version',
    'git status', 'git log --oneline -5', 'git log --stat',
    'git branch', 'git branch -a', 'git diff HEAD', 'git diff --cached',
    'git stash list', 'git remote -v', 'git fetch --dry-run',
    'pip list', 'pip show numpy', 'pip show requests',
    'conda list', 'conda info',
    'npm list', 'npm outdated', 'yarn list',
    'docker ps', 'docker images', 'docker stats --no-stream',
    'kubectl get pods', 'kubectl get nodes', 'kubectl describe pod app',
    'curl https://api.example.com', 'curl -I https://google.com',
    'wget --spider https://example.com',
    'ping -c 4 google.com', 'traceroute google.com',
    'grep "erro" file.log', 'grep -r "TODO" ./src',
    'grep -n "error" /var/log/app.log',
    'find /home -name "*.txt"', 'find /tmp -type f',
    'wc -l file.txt', 'wc -w document.txt',
    'cp file.txt /tmp/backup.txt', 'cp -r /etc /tmp/etc_backup',
    'mkdir /tmp/test_dir', 'mkdir -p /opt/app/logs',
    'touch new_file.txt', 'touch /tmp/.lock',
    'ln -s /var/app /opt/app',
    'diff file1.txt file2.txt', 'diff -u old.py new.py',
    'sort data.csv', 'uniq -c words.txt', 'cut -d"," -f1 data.csv',
    'awk "{print $1}" log.txt', 'sed "s/foo/bar/g" file.txt',
    'jq ".name" package.json',
    'ssh-keygen -l -f ~/.ssh/id_rsa.pub',
    'ping 8.8.8.8 -c 3', 'nslookup google.com',
    'systemctl status nginx', 'systemctl status mysql',
    'journalctl -u nginx --since today',
    'crontab -l', 'at -l',
    'terraform plan', 'terraform validate', 'terraform show',
    'kubectl get deployments', 'kubectl get services', 'kubectl logs app',
    'aws s3 ls', 'aws ec2 describe-instances', 'gcloud compute instances list',
]  # 155 exemplos

medio_risco = [
    # Permissões e usuários
    'chmod 777 script.sh', 'chmod 755 /var/www/html',
    'chmod 600 /etc/ssh/sshd_config', 'chmod +x deploy.sh',
    'chown -R www-data /var/www/html', 'chown ubuntu:ubuntu /opt/app',
    'useradd novo_usuario', 'useradd -m -s /bin/bash developer',
    'usermod -aG sudo novo_usuario', 'usermod -aG docker jenkins',
    'groupadd devteam', 'groupadd deploy',
    'passwd usuario_teste', 'passwd -e admin',
    # SQL
    'SELECT * FROM clientes', 'SELECT * FROM table_test',
    'SELECT COUNT(*) FROM pedidos WHERE status="pendente"',
    'UPDATE users SET active=0 WHERE id=5',
    'UPDATE clientes SET ativo=0 WHERE id=42',
    'UPDATE produtos SET preco=preco*1.1',
    'UPDATE estoque SET quantidade=quantidade-1 WHERE produto_id=10',
    'UPDATE config SET valor="novo" WHERE chave="timeout"',
    'INSERT INTO audit_log VALUES ("acao", NOW())',
    'INSERT INTO usuarios (nome, email) VALUES ("Ana", "ana@test.com")',
    'DELETE FROM logs WHERE date < "2023-01-01"',
    'DELETE FROM sessions WHERE expires < NOW()',
    'DELETE FROM cache WHERE created_at < "2022-01-01"',
    'ALTER TABLE logs ADD COLUMN origem VARCHAR(100)',
    'ALTER TABLE pedidos ADD COLUMN status_pagamento VARCHAR(20)',
    'ALTER TABLE usuarios ADD COLUMN last_login TIMESTAMP',
    'CREATE INDEX idx_email ON users(email)',
    'CREATE TABLE temp_import LIKE clientes',
    # Processos e serviços
    'kill -9 1234', 'kill -15 5678', 'kill -HUP 9999',
    'pkill -f servidor_antigo', 'pkill nginx',
    'systemctl restart nginx', 'systemctl restart php-fpm',
    'systemctl stop apache2', 'systemctl start redis',
    'service mysql restart', 'service cron reload',
    # Rede
    'iptables -A INPUT -p tcp --dport 8080 -j ACCEPT',
    'ufw allow 22/tcp', 'ufw allow 80/tcp',
    # Agendamento
    'crontab -e', 'at now + 1 hour',
    'nohup python worker.py &', 'screen -S sessao',
    'tmux new-session -d -s main', 'nice -n 10 python processo.py',
    'renice -n 5 -p 1234', 'ulimit -n 4096',
    # Sistema de arquivos
    'mount /dev/sdb1 /mnt/dados', 'umount /mnt/dados',
    'find /home -name "*.log" -mtime +30 -delete',
    'find /tmp -name "*.pid" -delete',
    'tar -czf backup.tar.gz /var/www/html',
    'tar -czf /backup/db_$(date +%Y%m%d).tar.gz /var/lib/mysql',
    'zip -r archive.zip /opt/projeto',
    'rsync -av /home/user/ /backup/user/',
    'rsync -avz --delete /var/www/ backup@server:/var/www/',
    'scp arquivo.txt user@servidor:/home/user/',
    # Configuração
    'sed -i "s/DEBUG=True/DEBUG=False/" settings.py',
    'sed -i "s/localhost/db.prod/" config.ini',
    # Docker e containers
    'docker stop my_container', 'docker restart nginx',
    'docker exec -it app bash', 'docker logs --tail 100 app',
    'docker-compose restart', 'docker network prune',
    # Kubernetes
    'kubectl rollout restart deployment/app',
    'kubectl scale deployment app --replicas=3',
    'kubectl exec -it pod-xyz -- bash',
    # Terraform
    'terraform apply -target=module.app',
    'terraform import aws_instance.web i-1234567890',
    # CI/CD
    'git push origin main', 'git merge feature/login',
    'git rebase main', 'git cherry-pick abc123',
    'npm run build', 'npm run test',
    'pytest tests/ -v', 'python manage.py migrate',
    'docker build -t app:latest .', 'docker push registry/app:1.0',
    'ansible-playbook deploy.yml --check', 'ansible-playbook deploy.yml',
    'helm upgrade app ./chart --dry-run', 'helm upgrade app ./chart',
    # Segurança não-destrutiva
    'openssl genrsa -out key.pem 2048',
    'openssl req -new -x509 -key key.pem -out cert.pem',
    'ssh-keygen -t ed25519 -C "deploy@server"',
    'gpg --gen-key', 'gpg --list-keys',
    'vault kv get secret/app', 'vault kv put secret/app key=value',
    'aws iam list-users', 'aws iam get-user --user-name admin',
    'gcloud iam service-accounts list',
    # Banco de dados não-destrutivo
    'ANALYZE TABLE pedidos', 'OPTIMIZE TABLE logs',
    'EXPLAIN SELECT * FROM users WHERE email="a@b.com"',
    'SHOW TABLES', 'SHOW PROCESSLIST', 'SHOW STATUS',
    'pg_dump -U postgres mydb > backup.sql',
    'mysqldump --no-data mydb > schema.sql',
    # Complementos v5
    'UPDATE pagamentos SET status="processado" WHERE id=77',
    'INSERT INTO eventos (tipo, ts) VALUES ("login", NOW())',
    'SELECT * FROM relatorios WHERE ano=2024',
    'ALTER TABLE produtos ADD COLUMN desconto DECIMAL(5,2)',
    'CREATE TABLE temp_pedidos AS SELECT * FROM pedidos WHERE 1=0',
    'mysqldump mydb > full_backup.sql',
    'pg_dump -Fc mydb -f mydb.dump',
    'SHOW CREATE TABLE usuarios',
    'EXPLAIN ANALYZE SELECT * FROM orders WHERE user_id=1',
    'DELETE FROM temp_data WHERE processed=1',
    'INSERT INTO notifications (user_id, msg) VALUES (1, "alerta")',
    'UPDATE sessions SET active=0 WHERE last_seen < "2023-06-01"',
    'ALTER TABLE clientes ADD COLUMN telefone VARCHAR(20)',
    'CREATE INDEX idx_status ON pedidos(status)',
]  # 123 exemplos

alto_risco = [
    # rm -rf em diretórios não-raiz
    'rm -rf /tmp/dados_antigos', 'rm -rf /tmp/logs',
    'rm -rf /home/user/projetos', 'rm -rf /opt/app/uploads',
    'rm -rf /var/log/old', 'rm -rf /srv/dados',
    'rm -rf /var/cache/app', 'rm -rf /tmp/uploads_2022',
    'rm -rf /opt/backups/2021', 'rm -rf /home/deploy/releases/old',
    'sudo rm -rf /var/log', 'sudo rm -rf /etc/nginx/sites-enabled',
    # DROP TABLE
    'DROP TABLE clientes', 'DROP TABLE users_production',
    'DROP TABLE pedidos', 'DROP TABLE vendas',
    'DROP TABLE clientes_historico', 'DROP TABLE relatorios',
    'DROP TABLE tokens_acesso', 'DROP TABLE configuracoes',
    'DROP TABLE permissoes', 'DROP TABLE integracoes',
    'DROP TABLE auditoria_financeira', 'DROP TABLE historico_precos',
    # DELETE amplo
    'DELETE FROM usuarios', 'DELETE FROM prod.users',
    'DELETE FROM financeiro WHERE ano < 2020',
    'DELETE FROM backups', 'DELETE FROM eventos',
    'DELETE FROM chaves_api', 'DELETE FROM webhooks',
    'DELETE FROM transacoes WHERE ano=2021',
    # TRUNCATE
    'TRUNCATE TABLE session_logs', 'TRUNCATE TABLE auditoria',
    'TRUNCATE TABLE logs_sistema', 'TRUNCATE TABLE metricas',
    'TRUNCATE TABLE notificacoes', 'TRUNCATE TABLE sessoes_usuario',
    # ALTER DROP COLUMN
    'ALTER TABLE users DROP COLUMN senha',
    'ALTER TABLE usuarios DROP COLUMN hash_senha',
    'ALTER TABLE pagamentos DROP COLUMN dados_cartao',
    # Docker destrutivo
    'docker rm -f $(docker ps -aq)', 'docker rmi -f $(docker images -q)',
    'docker volume rm $(docker volume ls -q)',
    'docker-compose down --volumes',
    # Kubernetes
    'kubectl delete deployment app', 'kubectl delete namespace staging',
    'kubectl delete pvc data-volume',
    # Configuração crítica
    'rm -rf /etc/nginx', 'rm -rf /opt/app/config',
    'rm -f /etc/crontab',
    # Cloud
    'aws s3 rm s3://my-bucket/data/ --recursive',
    'gsutil rm -r gs://my-bucket/backups/2021',
    'terraform destroy -auto-approve -target=module.database',
    # Banco staging
    'DROP DATABASE test_backup', 'DROP DATABASE staging_db',
    'DROP DATABASE dev_replica', 'DROP SCHEMA app_schema CASCADE',
    # Cloud (novos v4/v5)
    'helm uninstall my-app', 'helm uninstall monitoring',
    'kubectl delete configmap app-config', 'kubectl delete secret app-secret',
    'aws rds delete-db-instance --db-instance-identifier staging-db --skip-final-snapshot',
    'aws elasticache delete-cache-cluster --cache-cluster-id staging-cache',
    'gcloud sql instances delete staging-db',
    'az sql db delete --name staging --server myserver --resource-group rg',
    'rm -rf /var/lib/postgresql/data/pg_wal',
    'rm -rf /var/lib/mysql/ibdata1',
    'find / -name "*.log" -delete',
    'find /etc -name "*.conf" -delete',
    'DROP TABLE IF EXISTS audit_trail',
    'DROP TABLE IF EXISTS access_log',
    'DELETE FROM users WHERE created_at < "2020-01-01"',
    'DELETE FROM orders WHERE status="cancelled"',
    'TRUNCATE TABLE inventory',
    'TRUNCATE TABLE price_history',
    'ALTER TABLE accounts DROP COLUMN credit_card_hash',
    'kubectl drain node-1 --ignore-daemonsets --delete-emptydir-data',
    'kubectl cordon node-2',
    # Complementos v5
    'DROP TABLE IF EXISTS sessions_old',
    'DROP TABLE IF EXISTS temp_migration',
    'DELETE FROM audit WHERE created_at < "2018-01-01"',
    'DELETE FROM logs_antigos',
    'DELETE FROM cache_entries WHERE expired=1',
    'TRUNCATE TABLE webhook_log',
    'TRUNCATE TABLE email_queue',
    'TRUNCATE TABLE failed_jobs',
    'TRUNCATE TABLE password_resets',
    'ALTER TABLE users DROP COLUMN legacy_token',
    'ALTER TABLE orders DROP COLUMN old_status',
    'ALTER TABLE products DROP COLUMN deprecated_field',
    'rm -rf /opt/releases/v1',
    'rm -rf /var/www/html/old',
    'rm -rf /srv/backups/2019',
    'kubectl delete deployment worker',
    'helm uninstall redis',
    'helm uninstall elasticsearch',
    'DROP TABLE IF EXISTS payment_history',
    'DROP TABLE IF EXISTS user_tokens',
    'DROP TABLE IF EXISTS api_keys',
    'DELETE FROM permissions WHERE role="guest"',
    'DELETE FROM subscriptions WHERE expired=1',
    'DELETE FROM notifications WHERE read_at IS NOT NULL',
    'DELETE FROM temp_files',
    'TRUNCATE TABLE api_logs',
    'TRUNCATE TABLE request_cache',
    'TRUNCATE TABLE migration_log',
    'TRUNCATE TABLE debug_traces',
    'ALTER TABLE customers DROP COLUMN old_address',
    'ALTER TABLE invoices DROP COLUMN legacy_id',
    'rm -rf /opt/app/tmp',
    'rm -rf /var/www/uploads/2020',
    'rm -rf /home/deploy/cache',
    'kubectl delete service backend-api',
    'kubectl delete ingress app-ingress',
    'docker-compose down --rmi all',
    'aws s3 rm s3://staging-assets/ --recursive',
    'gsutil rm -r gs://staging-bucket/temp/',
]  # 118 exemplos

critico = [
    # rm -rf raiz
    'rm -rf /', 'rm -rf /* --no-preserve-root',
    'rm -rf /boot', 'rm -rf /etc', 'rm -rf /lib',
    'rm -rf /usr', 'rm -rf /var', 'rm -rf /sbin',
    'sudo rm -rf /boot', 'sudo rm -rf /etc',
    # dd / formatação
    'sudo dd if=/dev/zero of=/dev/sda',
    'sudo dd if=/dev/urandom of=/dev/sda bs=1M',
    'sudo dd if=/dev/zero of=/dev/nvme0n1',
    'sudo mkfs.ext4 /dev/sda1', 'sudo mke2fs /dev/sda1',
    'sudo wipefs -a /dev/sda', 'sudo shred -vfz /dev/sda',
    'sudo parted /dev/sda rm 1', 'sudo fdisk /dev/sda',
    # DROP DATABASE produção
    'DROP DATABASE production', 'DROP DATABASE empresa_master',
    'DROP DATABASE analytics_prod', 'DROP DATABASE financeiro_producao',
    'DROP DATABASE clientes_global', 'DROP DATABASE rh_corporativo',
    'DROP DATABASE erp_producao', 'DROP DATABASE bi_master',
    'DROP DATABASE ecommerce_producao', 'DROP DATABASE infraestrutura',
    'DROP DATABASE datawarehouse', 'DROP DATABASE crm_producao',
    'DROP DATABASE seguranca', 'DROP DATABASE compliance_dados',
    'DROP DATABASE master_producao', 'DROP DATABASE auth_service',
    # DROP SCHEMA
    'DROP SCHEMA public CASCADE', 'DROP SCHEMA app CASCADE',
    # SQL combinado
    'TRUNCATE TABLE users; DROP TABLE users',
    'DELETE FROM information_schema.tables',
    # Cloud catastrófico
    'terraform destroy -auto-approve',
    'kubectl delete namespace production',
    'kubectl delete all --all -n production',
    'kubectl delete pv --all',
    'helm uninstall $(helm list -q)',
    # Outros
    'iptables -F && iptables -X && iptables -P INPUT DROP',
    'sudo rm -rf /etc/ssl && sudo rm -rf /etc/ssh',
    'truncate -s 0 /etc/passwd',
    'sudo dd if=/dev/zero of=/dev/sda bs=1M count=512',
    'DROP DATABASE mysql', 'DROP DATABASE postgres',
    'sudo shred -n 3 /dev/nvme0n1',
    'sudo mkfs.xfs /dev/sdb',
    'aws ec2 terminate-instances --instance-ids $(aws ec2 describe-instances --query Reservations[].Instances[].InstanceId --output text)',
    'gcloud compute instances delete --zone us-central1-a $(gcloud compute instances list --format="value(name)")',
    # Cloud destrutivo adicional
    'DROP DATABASE IF EXISTS production',
    'DROP DATABASE IF EXISTS master',
    'sudo rm -rf /home', 'sudo rm -rf /root',
    'sudo rm -rf /tmp && sudo rm -rf /var/tmp',
    'aws s3 rm s3://prod-backups/ --recursive',
    'aws s3 rb s3://prod-data --force',
    'gcloud storage rm -r gs://prod-backups/**',
    'az storage account delete --name prodstorageaccount --resource-group prod-rg',
    'aws rds delete-db-instance --db-instance-identifier prod-db --skip-final-snapshot',
    'gcloud sql instances delete prod-db --quiet',
    'kubectl delete namespace kube-system',
    'kubectl delete clusterrole cluster-admin',
    'helm uninstall ingress-nginx -n ingress-nginx',
    'sudo systemctl disable --now sshd && sudo iptables -F',
    'cat /dev/null > /etc/shadow',
    'echo "" > /etc/passwd',
    'sudo kill -9 1', 'sudo pkill -9 -u root',
    # Complementos v5
    'DROP DATABASE portal_producao',
    'DROP DATABASE vendas_producao',
    'DROP DATABASE logs_producao',
    'DROP DATABASE relatorios_producao',
    'sudo rm -rf /bin',
    'sudo rm -rf /lib64',
    'sudo dd if=/dev/zero of=/dev/sdb bs=4M',
    'sudo wipefs -a /dev/nvme0n1',
    'kubectl delete all --all --all-namespaces',
    'aws ec2 terminate-instances --instance-ids i-0123456789abcdef0',
    'gcloud compute instances delete prod-server-1 --zone us-east1-b --quiet',
    'terraform destroy -auto-approve -var="env=production"',
    'DROP SCHEMA production CASCADE',
    'DROP SCHEMA billing CASCADE',
    'sudo rm -rf /etc/kubernetes',
    'sudo kubeadm reset --force',
    'DROP DATABASE IF EXISTS analytics_prod',
    'DROP DATABASE IF EXISTS bi_producao',
    'DROP DATABASE IF EXISTS erp_backup',
    'sudo rm -rf /proc',
    'sudo rm -rf /sys',
    'sudo dd if=/dev/urandom of=/dev/sdc bs=1M',
    'sudo wipefs -a /dev/sdc',
    'sudo shred -n 5 /dev/sda',
    'kubectl delete namespace monitoring',
    'kubectl delete namespace logging',
    'terraform destroy -auto-approve -target=aws_db_instance.prod',
    'aws s3 rb s3://company-backups --force',
    'gcloud sql instances delete prod-mysql --quiet',
    'sudo systemctl stop kubelet && sudo rm -rf /etc/kubernetes',
    'DROP SCHEMA analytics CASCADE',
    'sudo kill -9 -1',
    'truncate -s 0 /etc/shadow',
]  # 106 exemplos

# ── Verificação de consistência ───────────────────────────────────────────────
contagens = {
    '🟢 Baixo Risco': len(baixo_risco),
    '🟡 Médio Risco': len(medio_risco),
    '🟠 Alto Risco':  len(alto_risco),
    '🔴 Crítico':     len(critico),
}
total_real = sum(contagens.values())

print('📊 Distribuição do Dataset de Treino (v5.0):')
for label, count in contagens.items():
    bar = '█' * (count // 2)
    print(f'  {label:20s} | {bar} ({count})')
print(f'\n  Total real (verificado): {total_real} exemplos')

TOTAL_ESPERADO = 502
assert total_real == TOTAL_ESPERADO, (
    f'⚠️  Discrepância: {total_real} ≠ {TOTAL_ESPERADO}. Verifique as listas.'
)
print(f'  ✅ Consistência verificada: {total_real} == {TOTAL_ESPERADO}')

print('\n📈 Evolução histórica do dataset:')
for ver, total in [('v1.0',15),('v2.0',54),('v3.0',180),('v4.0',344),('v5.0',total_real)]:
    bar = '▓' * (total // 10)
    print(f'  {ver}: {bar} ({total})')

data_v5 = {
    'comando': baixo_risco + medio_risco + alto_risco + critico,
    'label':   [0]*len(baixo_risco) + [1]*len(medio_risco) +
               [2]*len(alto_risco)  + [3]*len(critico),
}
df_train  = pd.DataFrame(data_v5)
mapa_labels = {0: '🟢 Baixo Risco', 1: '🟡 Médio Risco',
               2: '🟠 Alto Risco',  3: '🔴 Crítico'}


## 🏗 4. Motor Principal — BlastRadiusEngine v5.0

### Arquitetura de 3 Camadas com Votação Ponderada

```
Comando de entrada
│
├──► Camada 1: Heurística de Palavras-Chave     (peso: 0.50)
│    Dicionário de termos com scores ponderados
│
├──► Camada 2: Random Forest Calibrado (TF-IDF) (peso: 0.25)
│    Vetorização com n-grams (1,3) e 1500 features
│
└──► Camada 3: XLM-RoBERTa-XNLI (Zero-Shot)    (peso: 0.25)
     Compreensão semântica multilingual
     + Threshold de confiança mínima (50%)
                │
                ▼
     Score Final Ponderado
     + Alerta de Divergência (diff > 1 AND confiança baixa)
     + Feedback Loop com re-treino incremental
```

### Mudanças (v4 → v5)

| Camada | v4.0 | v5.0 | Justificativa |
|---|---|---|---|
| Heurística | 50% | 50% | Mantida — determinística e auditável |
| Random Forest | 25% | 25% | Consolidado — dataset maior |
| NLP Semântico | 25% | 25% | Mantido — threshold previne falsos positivos |

### Novidades da v5

- **Feedback loop com re-treino real** — `retreinar_com_feedback()` reinjecta as
  correções no RF sem precisar re-executar toda a pipeline
- **Exportação CSV** — `exportar_feedback_csv()` gera base para pipeline de dados
- **Novos termos heurísticos** — `KILL -9 1`, `AWS RDS DELETE`, `HELM UNINSTALL`,
  `AWS S3 RB`, `GCLOUD SQL INSTANCES DELETE`


In [ ]:
class BlastRadiusEngine:
    """
    Motor de detecção de Raio de Impacto v5.0.

    Melhorias sobre v4.0:
    - Dataset de 502 exemplos com assertion de consistência (corrige discrepância v4)
    - Feedback loop com re-treino incremental real via retreinar_com_feedback()
    - Exportação do histórico de feedback para CSV
    - Novos termos heurísticos: AWS RDS, GCloud SQL, Helm, Kill PID 1, AWS S3 RB
    - Auditoria: cada predição registra timestamp, votos e confiança das 3 camadas
    """

    def __init__(self):
        self.keywords = {
            # Crítico
            'RM -RF /':                       100,
            '--NO-PRESERVE-ROOT':             100,
            'DD ':                             90,
            'DROP DATABASE':                   90,
            'DROP SCHEMA':                     90,
            'MKFS':                            90,
            'SHRED':                           90,
            'WIPEFS':                          85,
            'TERRAFORM DESTROY':               85,
            'KUBECTL DELETE ALL':              85,
            'TRUNCATE -S 0 /ETC':              90,
            'SUDO KILL -9 1':                  90,   # novo v5
            'CAT /DEV/NULL > /ETC/SHADOW':     95,   # novo v5
            'TRUNCATE -S 0 /ETC/SHADOW':       95,   # novo v5
            # Alto Risco
            'RM -RF':                          60,
            'DROP TABLE':                      55,
            'TRUNCATE TABLE':                  50,
            'TRUNCATE':                        50,
            'DELETE FROM':                     45,
            'DOCKER RM -F':                    45,
            'DOCKER RMI -F':                   45,
            'KUBECTL DELETE':                  40,
            'HELM UNINSTALL':                  45,   # novo v5
            'AWS RDS DELETE':                  55,   # novo v5
            'GCLOUD SQL INSTANCES DELETE':     55,   # novo v5
            'AWS S3 RB':                       50,   # novo v5
            # Médio Risco
            'DROP':                            40,
            'DELETE':                          35,
            'UPDATE':                          25,
            'ALTER TABLE':                     25,
            'CHMOD 777':                       20,
            'KILL -9':                         20,
            'PROD':                            15,
            'PRODUCTION':                      15,
            # Contexto
            'DATABASE':                        10,
            'SUDO':                            10,
            'ALL':                              5,
        }
        self.pesos = {'heuristica': 0.50, 'rf': 0.25, 'nlp': 0.25}
        self.feedback_db      = {}
        self.feedback_history = []

        self.vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=1500)
        self.model_rf   = RandomForestClassifier(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            class_weight='balanced', random_state=42
        )
        self._treinar_modelo_base()

    def _treinar_modelo_base(self):
        X = self.vectorizer.fit_transform(df_train['comando'])
        y = df_train['label']
        self.model_rf.fit(X, y)

    # ── Feedback Loop ──────────────────────────────────────────────────────────
    def registrar_feedback(self, comando, label_correto, label_predito):
        """Registra correção do operador (override imediato via lookup exato)."""
        key = comando.upper().strip()
        self.feedback_db[key] = label_correto
        self.feedback_history.append({
            'timestamp':     datetime.datetime.now().isoformat(),
            'comando':       comando,
            'label_predito': label_predito,
            'label_correto': label_correto,
        })
        print(f'✅ Feedback registrado: "{comando}" → label {label_correto}')

    def retreinar_com_feedback(self):
        """
        Re-treina o Random Forest incorporando as correções do feedback loop.
        Permite aprendizado incremental sem re-treino completo da pipeline.
        """
        if not self.feedback_history:
            print('⚠️  Nenhum feedback registrado. Re-treino ignorado.')
            return
        novos_cmds   = [e['comando']       for e in self.feedback_history]
        novos_labels = [e['label_correto'] for e in self.feedback_history]
        X_orig    = self.vectorizer.transform(df_train['comando'])
        X_novos   = self.vectorizer.transform(novos_cmds)
        X_combined = sp.vstack([X_orig, X_novos])
        y_combined = list(df_train['label']) + novos_labels
        self.model_rf.fit(X_combined, y_combined)
        print(f'✅ Re-treinado com {len(novos_cmds)} correção(ões). '
              f'Dataset total: {len(y_combined)} exemplos.')

    def exportar_feedback_csv(self, caminho='feedback_v5.csv'):
        """Exporta o histórico de feedback para CSV (base de re-treino futuro)."""
        if not self.feedback_history:
            print('⚠️  Nenhum feedback para exportar.')
            return
        output = StringIO()
        writer = csv.DictWriter(
            output,
            fieldnames=['timestamp','comando','label_predito','label_correto']
        )
        writer.writeheader()
        writer.writerows(self.feedback_history)
        try:
            with open(caminho, 'w', newline='', encoding='utf-8') as f:
                f.write(output.getvalue())
            print(f'✅ Exportado: {caminho} ({len(self.feedback_history)} entradas)')
        except Exception:
            print('CSV (saída em memória):')
            print(output.getvalue())

    # ── Heurística ─────────────────────────────────────────────────────────────
    def calcular_heuristica(self, comando):
        cmd_up = comando.upper()
        score, termos = 0, []
        for term, peso in self.keywords.items():
            if term in cmd_up:
                score += peso
                termos.append(term.strip())
        return min(score, 100), termos

    def categorizar_texto(self, score):
        if score < 30: return '🟢 Baixo Risco (Impacto Restrito)'
        if score < 55: return '🟡 Médio Risco (Atenção Necessária)'
        if score < 75: return '🟠 Alto Risco (Bloqueio Recomendado)'
        return '🔴 Crítico (Risco Catastrófico)'

    def inferir_alvos(self, comando, classe_risco):
        alvos, cmd_up = [], comando.upper()
        mapa = {
            'Database Master':         ['DATABASE', 'DB '],
            'Tabela Específica':       ['TABLE', 'TRUNCATE', 'INSERT INTO'],
            'Cluster de Produção':     ['PROD', 'PRODUCTION'],
            'Sistema de Autenticação': ['USER', 'USUARIO', 'AUTH', 'LOGIN', 'PASSWD'],
            'Sistema de Arquivos':     ['/DEV/', 'MKFS', 'DD ', 'RM -RF', 'SHRED'],
            'Camada de API':           ['API', 'CURL', 'HTTP', 'ENDPOINT'],
            'Serviço de Logs':         ['LOG', '/VAR/LOG'],
            'Serviços Dependentes':    ['CASCADE', 'FOREIGN'],
            'Infraestrutura Cloud':    ['TERRAFORM', 'AWS ', 'GCP ', 'GCLOUD', 'AZURE', 'AZ '],
            'Cluster Kubernetes':      ['KUBECTL', 'HELM ', 'NAMESPACE', 'KUBE'],
            'Containers Docker':       ['DOCKER', 'DOCKER-COMPOSE'],
            'Pipeline CI/CD':          ['GITHUB', 'GITLAB', 'JENKINS', 'ANSIBLE'],
            'Segredos e Credenciais':  ['SECRET', 'VAULT', 'SSH', 'SSL', 'SHADOW'],
        }
        for alvo, triggers in mapa.items():
            if any(t in cmd_up for t in triggers):
                alvos.append(alvo)
        if classe_risco >= 3 and not alvos:
            alvos = ['Cloud Provider', 'Auth System', 'Database Master']
        return alvos if alvos else ['Ambiente Local']

    # ── Análise principal ──────────────────────────────────────────────────────
    def analisar_comando(self, comando):
        # 1. Feedback override
        chave = comando.upper().strip()
        if chave in self.feedback_db:
            co = self.feedback_db[chave]
            sv = [15, 42, 65, 90][co]
            return {
                'comando': comando, 'classe_final': co, 'score_visual': sv,
                'categoria_visual': self.categorizar_texto(sv),
                'termos': ['[FEEDBACK OVERRIDE]'], 'divergencia': False,
                'votos': [co, co, co],
                'detalhe': {
                    'heuristica':    {'label': co, 'score_raw': sv},
                    'random_forest': {'label': co, 'confianca': 100.0},
                    'nlp':           {'label': co, 'insight': 'feedback override',
                                      'confianca': 100.0},
                },
                'alvos': self.inferir_alvos(comando, co),
                'feedback_applied': True,
            }

        # 2. Heurística
        score_h, termos = self.calcular_heuristica(comando)
        cat_h = 0 if score_h < 30 else 1 if score_h < 55 else 2 if score_h < 75 else 3

        # 3. Random Forest
        X_input  = self.vectorizer.transform([comando])
        pred_rf  = int(self.model_rf.predict(X_input)[0])
        proba_rf = self.model_rf.predict_proba(X_input)[0]
        conf_rf  = round(float(max(proba_rf)) * 100, 1)

        # 4. NLP com threshold de confiança
        res_hf       = classifier_hf(comando, labels_hf)
        label_nlp_en = res_hf['labels'][0]
        conf_nlp     = round(res_hf['scores'][0] * 100, 1)
        label_nlp    = mapa_labels_display.get(label_nlp_en, label_nlp_en)
        mapa_nlp     = {'low risk': 0, 'medium risk': 1, 'high risk': 2, 'critical': 3}

        if conf_nlp < NLP_CONFIDENCE_THRESHOLD * 100:
            pred_nlp  = cat_h
            label_nlp = (f'abstido ({conf_nlp:.0f}% < '
                         f'threshold {NLP_CONFIDENCE_THRESHOLD*100:.0f}%)')
        else:
            pred_nlp = mapa_nlp.get(label_nlp_en, 0)

        # 5. Votação ponderada
        score_final  = (cat_h   * self.pesos['heuristica'] +
                        pred_rf  * self.pesos['rf'] +
                        pred_nlp * self.pesos['nlp'])
        classe_final = min(3, round(score_final))

        # 6. Divergência refinada
        votos       = [cat_h, pred_rf, pred_nlp]
        diff_votos  = max(votos) - min(votos)
        conf_baixa  = conf_rf < 60 or conf_nlp < NLP_CONFIDENCE_THRESHOLD * 100
        divergencia = diff_votos > 1 and conf_baixa

        score_visual = min(100, round(score_final / 3 * 100))

        return {
            'comando': comando, 'classe_final': classe_final,
            'score_visual': score_visual,
            'categoria_visual': self.categorizar_texto(score_visual),
            'termos': termos, 'divergencia': divergencia, 'votos': votos,
            'detalhe': {
                'heuristica':    {'label': cat_h,   'score_raw': score_h},
                'random_forest': {'label': pred_rf,  'confianca': conf_rf},
                'nlp':           {'label': pred_nlp, 'insight': label_nlp,
                                  'confianca': conf_nlp},
            },
            'alvos': self.inferir_alvos(comando, classe_final),
            'feedback_applied': False,
        }


print('⏳ Treinando BlastRadiusEngine v5.0...')
engine = BlastRadiusEngine()
print('✅ BlastRadiusEngine v5.0 pronto!')
print(f'   Pesos: Heurística {engine.pesos["heuristica"]:.0%} | '
      f'RF {engine.pesos["rf"]:.0%} | NLP {engine.pesos["nlp"]:.0%}')
print(f'   Dataset de treino: {len(df_train)} exemplos')


## 📊 5. Validação do Modelo — Métricas, Cross-Validation e Curva de Aprendizado

### O que estamos avaliando

| Métrica | O que mede | Meta v5 |
|---|---|---|
| F1 weighted (CV 5-fold) | Generalização do RF | **≥ 0,75** |
| Desvio padrão CV | Estabilidade entre folds | ≤ 0,05 |
| Acurácia global | Corretude no treino | ≥ 92% |
| Gap treino / validação | Diagnóstico de overfitting | ≤ 0,10 |

### Curva de Aprendizado (novo na v5)

A curva de aprendizado é uma ferramenta essencial para diagnosticar **overfitting**.
O gap entre as curvas de treino e validação indica o quanto o modelo memoriza em vez
de generalizar. Uma curva de validação que converge para a de treino ao aumentar o
dataset confirma que mais dados continuam beneficiando o modelo.

> O Random Forest atua como camada de reforço (peso 25%). O ensemble com heurística
> e NLP compensa eventuais limitações do RF isolado. Para produção real,
> 1000+ exemplos e fine-tuning supervisionado do NLP seriam necessários.


In [ ]:
X_all = engine.vectorizer.transform(df_train['comando'])
y_all = df_train['label']

# ── Cross-Validation estratificado ───────────────────────────────────────────
skf       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_cv = cross_val_score(engine.model_rf, X_all, y_all, cv=skf,
                             scoring='f1_weighted')

print('🔁 Cross-Validation Estratificado (5-fold) — F1 Weighted:')
print(f'   Scores por fold : {[round(s, 3) for s in scores_cv]}')
print(f'   Média           : {scores_cv.mean():.3f}')
print(f'   Desvio Padrão   : {scores_cv.std():.3f}')
meta = '✅ META ATINGIDA' if scores_cv.mean() >= 0.75 else '⚠️  Abaixo da meta'
print(f'   Status (≥ 0,75) : {meta}')

# ── Comparação histórica ──────────────────────────────────────────────────────
print('\n📈 Evolução do F1 Weighted (CV 5-fold):')
evolucao = [
    ('v2.0', 0.479, 0.138),
    ('v3.0', 0.671, 0.052),
    ('v4.0', 0.689, 0.032),
    ('v5.0', scores_cv.mean(), scores_cv.std()),
]
for ver, media, std in evolucao:
    bar = '█' * int(media * 30)
    print(f'  {ver}: {bar} {media:.3f} ± {std:.3f}')

# ── Classification Report ─────────────────────────────────────────────────────
y_pred        = engine.model_rf.predict(X_all)
nomes_classes = ['Baixo Risco', 'Médio Risco', 'Alto Risco', 'Crítico']
print('\n📋 Classification Report (treino completo):')
print(classification_report(y_all, y_pred, target_names=nomes_classes))

# ── Diagnóstico de Overfitting ────────────────────────────────────────────────
acc_treino = (y_pred == y_all).mean()
gap        = acc_treino - scores_cv.mean()
status_gap = '✅ Aceitável' if gap <= 0.15 else '⚠️  Possível overfitting'
print(f'🔍 Diagnóstico de Overfitting:')
print(f'   Acurácia no treino  : {acc_treino:.3f}')
print(f'   F1 médio CV         : {scores_cv.mean():.3f}')
print(f'   Gap (treino - CV)   : {gap:.3f} → {status_gap}')

# ── Curva de Aprendizado ──────────────────────────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    engine.model_rf, X_all, y_all,
    cv=skf, scoring='f1_weighted',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

# ── Visualizações ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Matriz de Confusão
cm   = confusion_matrix(y_all, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=nomes_classes)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — Random Forest v5.0\n(Dataset de Treino)')
axes[0].set_xticklabels(nomes_classes, rotation=20, ha='right', fontsize=9)
axes[0].set_yticklabels(nomes_classes, fontsize=9)

# 2. Evolução F1 por versão
vers   = ['v2.0', 'v3.0', 'v4.0', 'v5.0']
medias = [0.479, 0.671, 0.689, scores_cv.mean()]
stds   = [0.138, 0.052, 0.032, scores_cv.std()]
cores  = ['#e74c3c', '#f39c12', '#3498db', '#27ae60']
bars   = axes[1].bar(vers, medias, color=cores, width=0.5, edgecolor='white', linewidth=1.5)
axes[1].errorbar(vers, medias, yerr=stds, fmt='none', color='black', capsize=5, linewidth=1.5)
for bar, val, std in zip(bars, medias, stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.02,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('F1 Weighted (CV 5-fold)')
axes[1].set_title('Evolução do F1 Weighted por Versão\n(barras de erro = desvio padrão)')
axes[1].axhline(0.75, color='green', linestyle='--', alpha=0.6, label='Meta v5 (0,75)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# 3. Curva de Aprendizado
axes[2].plot(train_sizes, train_mean, 'o-', color='#2c3e50', label='Treino')
axes[2].fill_between(train_sizes, train_mean-train_std, train_mean+train_std,
                     alpha=0.15, color='#2c3e50')
axes[2].plot(train_sizes, val_mean, 'o-', color='#27ae60', label='Validação (CV)')
axes[2].fill_between(train_sizes, val_mean-val_std, val_mean+val_std,
                     alpha=0.15, color='#27ae60')
axes[2].axhline(0.75, color='red', linestyle='--', alpha=0.5, label='Meta F1 = 0,75')
axes[2].set_xlabel('Tamanho do dataset de treino')
axes[2].set_ylabel('F1 Weighted')
axes[2].set_title('Curva de Aprendizado — Random Forest v5.0\n(diagnóstico de overfitting)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 🧪 6. Validação — Bateria de Testes Estruturados

Submetemos o motor a **15 comandos** cobrindo todos os níveis de risco,
incluindo os casos problemáticos herdados das versões anteriores e os novos
casos cloud-native da v5.

| Comando | Problema anterior | Correção |
|---|---|---|
| `ls -la /home` | NLP chamava de crítico (41% conf.) | Threshold 50% → abstido |
| `git status` | NLP chamava de crítico | Threshold 50% → abstido |
| `UPDATE clientes SET ativo=0 WHERE id=42` | Divergência na v3 | Dataset maior → consenso |
| `kubectl delete namespace staging` | Ausente na v3 | Cobertura cloud v4/v5 |
| `aws rds delete-db-instance ...` | Novo na v5 | Heurística AWS RDS DELETE |


In [ ]:
comandos_teste = [
    # Casos problemáticos herdados
    'ls -la /home',
    'git status',
    # Operações inofensivas
    'cat /etc/hosts',
    'docker ps',
    'kubectl get pods',
    'terraform plan',
    # Médio risco
    'UPDATE clientes SET ativo=0 WHERE id=42',
    'chmod 777 /var/www/html',
    'systemctl restart nginx',
    # Alto risco
    'rm -rf /tmp/dados_antigos',
    'DROP TABLE users_production',
    'kubectl delete namespace staging',
    # Crítico
    'DROP DATABASE production',
    'terraform destroy -auto-approve',
    'aws rds delete-db-instance --db-instance-identifier prod-db --skip-final-snapshot',
]

resultados = [engine.analisar_comando(c) for c in comandos_teste]

rows = []
for r in resultados:
    nlp_info = r['detalhe']['nlp']
    conf_nlp = nlp_info.get('confianca', 0)
    abstido  = conf_nlp < NLP_CONFIDENCE_THRESHOLD * 100
    rows.append({
        'Comando':     r['comando'],
        'Score':       r['score_visual'],
        'NLP Insight': nlp_info['insight'],
        'NLP Conf%':   f'{conf_nlp:.0f}%',
        'NLP Abstido': '✅' if abstido else '—',
        'Categoria':   r['categoria_visual'],
        'Divergência': 'SIM' if r['divergencia'] else 'não',
    })

df_result = pd.DataFrame(rows)
display(df_result)

total       = len(resultados)
divergentes = sum(1 for r in resultados if r['divergencia'])
abstidos    = sum(1 for r in resultados
                  if r['detalhe']['nlp'].get('confianca', 100) < NLP_CONFIDENCE_THRESHOLD * 100)

print(f'\n📌 Resumo da bateria de testes:')
print(f'   Total de comandos   : {total}')
print(f'   Com divergência     : {divergentes} ({divergentes/total*100:.1f}%)')
print(f'   NLP abstido (< 50%) : {abstidos} ({abstidos/total*100:.1f}%)')


## 🔬 6b. Análise Crítica — Divergência, Limitações e Evolução

Esta seção documenta com honestidade os pontos fortes, as limitações remanescentes
e as decisões de design do Blast Radius Detector v5.0.


In [ ]:
print('=' * 70)
print('📊 ANÁLISE CRÍTICA DO SISTEMA — Blast Radius Detector v5.0')
print('=' * 70)

pct_div     = round(divergentes / total * 100, 1)
reducao_v4  = round(30.8 - pct_div, 1)
reducao_v3  = round(55.6 - pct_div, 1)

print(f'''
Taxa de Divergência nos Testes v5.0:
  {divergentes} de {total} comandos acionaram alerta ({pct_div}%)
  v4.0 era 30,8% — redução adicional de {reducao_v4} pontos percentuais
  v3.0 era 55,6% — redução acumulada de {reducao_v3} pontos percentuais

NLP com Threshold de Confiança:
  {abstidos} de {total} comandos tiveram NLP abstido (conf < 50%)
  Previne penalização de comandos inofensivos como ls -la e git status

Melhorias implementadas na v5.0:
  [+] Dataset: 344 → 502 exemplos com assertion de consistência (corrige discrepância v4)
  [+] Feedback loop com re-treino incremental real (retreinar_com_feedback)
  [+] Exportação do histórico de feedback para CSV (base de re-treino futuro)
  [+] Curva de aprendizado para diagnóstico de overfitting
  [+] Novos termos heurísticos: AWS RDS DELETE, GCloud SQL, Helm, Kill PID 1
  [+] Novos exemplos cloud-native (CI/CD, segurança, banco não-destrutivo)
  [+] Auditoria completa: cada predição registra votos e confiança de cada camada

Limitações remanescentes (para v6.0):
  [-] Dataset ainda abaixo de 1000 exemplos (meta para produção real)
  [-] Heurística não captura contexto de caminho completo
  [-] Feedback loop baseado em lookup exato — não generaliza para variações
  [-] Fine-tuning supervisionado do NLP pendente para produção
  [-] Ausência de rate-limit e autenticação no console interativo
''')

# ── Visualizações ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Análise Crítica — Blast Radius Detector v5.0',
             fontsize=13, fontweight='bold')

# 1. Distribuição por categoria
categorias = [r['categoria_visual'].split('(')[0].strip() for r in resultados]
cat_count  = Counter(categorias)
cores_cat  = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
axes[0].pie(cat_count.values(), labels=cat_count.keys(),
            autopct='%1.0f%%', colors=cores_cat[:len(cat_count)],
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Distribuição por Categoria\n(Bateria de Testes v5)', fontweight='bold')

# 2. Evolução da divergência v3 → v5
labels_div = ['v3.0\n(55,6%)', 'v4.0\n(30,8%)', f'v5.0\n({pct_div}%)']
vals_div   = [55.6, 30.8, pct_div]
cores_div  = ['#e74c3c', '#f39c12', '#27ae60']
bars = axes[1].bar(labels_div, vals_div, color=cores_div, width=0.4,
                   edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, vals_div):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[1].set_ylim(0, 80)
axes[1].set_ylabel('Taxa de Divergência (%)')
axes[1].set_title('Redução da Divergência\nv3 → v4 → v5', fontweight='bold')
axes[1].axhline(20, color='green', linestyle='--', alpha=0.4, label='Meta ideal (20%)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# 3. Dataset v5
dist_treino = df_train['label'].value_counts().sort_index()
nomes_dist  = ['Baixo\nRisco', 'Médio\nRisco', 'Alto\nRisco', 'Crítico']
bars2 = axes[2].bar(nomes_dist, dist_treino.values,
                    color=['#2ecc71','#f1c40f','#e67e22','#e74c3c'],
                    edgecolor='white', linewidth=2)
for bar, val in zip(bars2, dist_treino.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', fontweight='bold', fontsize=11)
axes[2].set_title(f'Dataset de Treino v5.0\n({len(df_train)} exemplos — verificado)',
                  fontweight='bold')
axes[2].set_ylabel('Quantidade de exemplos')
axes[2].set_ylim(0, max(dist_treino.values) + 20)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 🔄 7. Feedback Loop — Aprendizado Incremental com Re-treino Real

### Evolução do Feedback Loop por versão

| Aspecto | v3.0 | v4.0 | v5.0 |
|---|---|---|---|
| Mecanismo | ❌ Inexistente | ✅ Lookup exato | ✅ Lookup + re-treino incremental |
| Exportação | ❌ | ❌ | ✅ CSV para pipeline de dados |
| Re-treino | ❌ | ❌ | ✅ `retreinar_com_feedback()` |
| Generalização | ❌ | ❌ | Parcial (RF atualizado) |

### Como funciona o ciclo completo

1. O operador identifica uma classificação incorreta no console
2. Registra a correção via `engine.registrar_feedback(cmd, correto, predito)`
3. Na próxima análise do **mesmo** comando, o override é aplicado imediatamente
4. Com `retreinar_com_feedback()`, o RF é re-treinado com os novos exemplos
5. O histórico pode ser exportado como CSV para re-treino futuro

> **Limitação conhecida:** o lookup exato não generaliza para variações do mesmo
> comando (e.g., corrigir `rm -rf /tmp/logs` não afeta `rm -rf /tmp/cache`).
> Generalização completa requereria fine-tuning periódico com os dados acumulados.


In [ ]:
print('=' * 65)
print('DEMONSTRAÇÃO DO FEEDBACK LOOP v5.0')
print('=' * 65)

# Caso 1: falso positivo (Critical → Alto)
cmd1      = 'rm -rf /tmp/logs'
res_antes = engine.analisar_comando(cmd1)
print(f'\n[CASO 1] Correção de falso positivo')
print(f'  Comando  : {cmd1}')
print(f'  ANTES    : {res_antes["categoria_visual"]} (score {res_antes["score_visual"]}/100)')
print(f'  Operador : label correto = 2 (Alto Risco — /tmp é não-crítico)')
engine.registrar_feedback(cmd1, label_correto=2, label_predito=res_antes['classe_final'])
res_depois = engine.analisar_comando(cmd1)
print(f'  DEPOIS   : {res_depois["categoria_visual"]} (score {res_depois["score_visual"]}/100)')
print(f'  Override : {res_depois.get("feedback_applied", False)}')

# Caso 2: falso negativo (Alto → Crítico por contexto)
cmd2       = 'kubectl delete namespace monitoring'
res_antes2 = engine.analisar_comando(cmd2)
print(f'\n[CASO 2] Correção de falso negativo (contexto de produção)')
print(f'  Comando  : {cmd2}')
print(f'  ANTES    : {res_antes2["categoria_visual"]} (score {res_antes2["score_visual"]}/100)')
print(f'  Operador : label correto = 3 (Crítico — monitoring é parte do stack de produção)')
engine.registrar_feedback(cmd2, label_correto=3, label_predito=res_antes2['classe_final'])
res_depois2 = engine.analisar_comando(cmd2)
print(f'  DEPOIS   : {res_depois2["categoria_visual"]} (score {res_depois2["score_visual"]}/100)')

# Re-treino incremental
print(f'\n[RE-TREINO] Incorporando {len(engine.feedback_history)} correção(ões) ao modelo...')
engine.retreinar_com_feedback()

# Histórico
mapa_nomes = {0: 'Baixo', 1: 'Médio', 2: 'Alto', 3: 'Crítico'}
print(f'\nHistórico de Feedback ({len(engine.feedback_history)} entrada(s)):')
for i, entry in enumerate(engine.feedback_history, 1):
    pred = mapa_nomes.get(entry['label_predito'], '?')
    corr = mapa_nomes.get(entry['label_correto'], '?')
    print(f'  [{i}] "{entry["comando"]}"')
    print(f'       Predito: {pred} → Correto: {corr} | {entry["timestamp"]}')

# Exportação
print()
engine.exportar_feedback_csv()
print('\nDica: acumule correções em engine.feedback_history para re-treino futuro.')


## 🌐 8. Grafo Dinâmico de Propagação do Raio de Impacto

### Evolução do grafo v1 → v5

| Aspecto | v1.0 | v3.0 | v5.0 |
|---|---|---|---|
| Nós de infra | Lista fixa | Detectados do comando | + Cloud, K8s, Docker, CI/CD, Credenciais |
| Cascata | Estática | Dinâmica | Nós críticos geram cascata automática |
| Categorias de alvos | 4 | 8 | **13** |

### Níveis de Propagação

- **Nível 0 — Local:** Impacto limitado ao ambiente de execução
- **Nível 1 — Setorial:** Afeta tabelas, logs ou arquivos de configuração
- **Nível 2 — Sistêmico:** Propaga para bancos de dados, APIs, storage
- **Nível 3 — Catastrófico:** Atinge o núcleo da infraestrutura ou cloud


In [ ]:
def plot_impact_graph(resultado):
    comando      = resultado['comando']
    classe_final = resultado['classe_final']
    alvos        = resultado['alvos']
    divergencia  = resultado['divergencia']

    cor_por_classe = {0: '#2ecc71', 1: '#f1c40f', 2: '#e67e22', 3: '#e74c3c'}
    cor_principal  = cor_por_classe.get(classe_final, '#95a5a6')

    G    = nx.DiGraph()
    root = 'EXECUÇÃO'
    G.add_node(root)

    nos_cascata = {
        'Database Master', 'Cluster de Produção',
        'Infraestrutura Cloud', 'Cluster Kubernetes',
        'Segredos e Credenciais',
    }
    for alvo in alvos:
        G.add_edge(root, alvo)
        if classe_final >= 2 and alvo in nos_cascata:
            G.add_edge(alvo, 'Cascata de Dependências')

    pos        = nx.spring_layout(G, seed=42, k=2.5)
    node_cols  = []
    for node in G.nodes():
        if node == root:             node_cols.append('#2c3e50')
        elif 'Cascata' in node:     node_cols.append('#c0392b')
        else:                       node_cols.append(cor_principal)

    plt.figure(figsize=(11, 5))
    nx.draw(G, pos, with_labels=True, node_color=node_cols, node_size=3500,
            font_size=8, font_color='white', font_weight='bold',
            edge_color='#7f8c8d', arrows=True, arrowsize=20, width=2)
    aviso_div = ' | DIVERGÊNCIA DETECTADA' if divergencia else ''
    fb_tag    = ' [FEEDBACK OVERRIDE]' if resultado.get('feedback_applied') else ''
    plt.title(
        f'Raio de Impacto — {resultado["categoria_visual"]}{aviso_div}{fb_tag}\n'
        f'Comando: {comando[:70]}{"..." if len(comando) > 70 else ""}',
        fontsize=11, fontweight='bold', pad=15
    )
    plt.tight_layout()
    plt.show()


for cmd in ['ls -la /home', 'DROP TABLE users_production',
            'DROP DATABASE production', 'terraform destroy -auto-approve']:
    res = engine.analisar_comando(cmd)
    plot_impact_graph(res)


## 🖥 9. Console Interativo de Análise de Risco

Interface integrada com:
- **Semáforo visual** baseado no score final ponderado
- **Alerta de divergência** com detalhamento dos votos por camada
- **Breakdown por camada** — heurística, RF e NLP com confiança
- **Indicador NLP abstido** quando confiança < 50%
- **Feedback loop inline** — campo de correção diretamente no console
- **Botão de re-treino** — incorpora correções ao RF sem reiniciar o notebook *(novo na v5)*
- **Grafo dinâmico** gerado automaticamente após cada análise

| Cor da Borda | Score | Significado |
|---|---|---|
| 🟢 Verde | < 30 | Seguro — Impacto restrito local |
| 🟡 Amarela | 30–54 | Atenção — Afeta estruturas específicas |
| 🟠 Laranja | 55–74 | Bloqueio recomendado |
| 🔴 Vermelha | ≥ 75 | Alerta crítico — Falha catastrófica |


In [ ]:
def cor_semaforo(score):
    if score < 30: return '#27ae60'
    if score < 55: return '#f39c12'
    if score < 75: return '#e67e22'
    return '#e74c3c'

_ultimo_resultado = {}

def processar_interacao(b):
    global _ultimo_resultado
    cmd = input_box.value.strip()
    if not cmd:
        return
    with saida:
        clear_output(wait=True)
        res = engine.analisar_comando(cmd)
        _ultimo_resultado = res
        cor = cor_semaforo(res['score_visual'])
        d   = res['detalhe']

        div_html = ''
        if res['divergencia']:
            votos_str = ' | '.join([
                f'Heurística: {d["heuristica"]["label"]}',
                f'RF: {d["random_forest"]["label"]}',
                f'NLP: {d["nlp"]["label"]}',
            ])
            div_html = (
                '<div style="background:#fff3cd;border:1px solid #ffc107;'
                'border-radius:6px;padding:8px;margin-top:8px;">'
                f'⚠️ <b>Divergência detectada:</b> {votos_str}</div>'
            )

        fb_html = ''
        if res.get('feedback_applied'):
            fb_html = (
                '<div style="background:#d4edda;border:1px solid #28a745;'
                'border-radius:6px;padding:8px;margin-top:8px;">'
                '✅ <b>Feedback override aplicado</b> — '
                'classificação corrigida pelo operador</div>'
            )

        nlp_conf    = d['nlp'].get('confianca', 100)
        nlp_abstido = nlp_conf < NLP_CONFIDENCE_THRESHOLD * 100
        nlp_badge   = (
            f'<span style="color:#e67e22;font-weight:bold">'
            f'⚠ Abstido ({nlp_conf:.0f}%)</span>'
            if nlp_abstido else
            f'<span style="color:#27ae60">✓ {nlp_conf:.0f}% confiança</span>'
        )

        termos_str = (
            ', '.join(f'<code>{t}</code>' for t in res['termos'])
            if res['termos'] else '<em>nenhum</em>'
        )

        display(HTML(f"""
<div style='border:3px solid {cor}; border-radius:12px; padding:16px;
     margin-top:12px; background:#fafafa; font-family:monospace;'>
  <h3 style='margin:0 0 10px 0; color:{cor};'>
    Análise de Risco:
    <code style='background:#eee;padding:2px 6px;border-radius:4px;'>
      {res['comando']}
    </code>
  </h3>
  <b>Resultado:</b> {res['categoria_visual']}<br>
  <b>Score Final:</b> {res['score_visual']}/100<br>
  <b>Termos detectados:</b> {termos_str}<br><br>
  <table style='width:100%;border-collapse:collapse;font-size:12px;'>
    <tr style='background:#eee;'>
      <th style='padding:6px;text-align:left;'>Camada</th>
      <th style='padding:6px;text-align:left;'>Resultado</th>
      <th style='padding:6px;text-align:left;'>Detalhe</th>
    </tr>
    <tr>
      <td style='padding:6px;'>🔑 Heurística (50%)</td>
      <td style='padding:6px;'>Label {d['heuristica']['label']}</td>
      <td style='padding:6px;'>Score bruto: {d['heuristica']['score_raw']}/100</td>
    </tr>
    <tr style='background:#f9f9f9;'>
      <td style='padding:6px;'>🌲 Random Forest (25%)</td>
      <td style='padding:6px;'>Label {d['random_forest']['label']}</td>
      <td style='padding:6px;'>Confiança: {d['random_forest']['confianca']:.1f}%</td>
    </tr>
    <tr>
      <td style='padding:6px;'>🧠 XLM-RoBERTa (25%)</td>
      <td style='padding:6px;'>{d['nlp']['insight']}</td>
      <td style='padding:6px;'>{nlp_badge}</td>
    </tr>
  </table>
  <br><b>Alvos detectados:</b> {', '.join(res['alvos'])}
  {div_html}
  {fb_html}
</div>
"""))
        plot_impact_graph(res)


def registrar_feedback_interativo(b):
    global _ultimo_resultado
    if not _ultimo_resultado:
        print('Analise um comando primeiro.')
        return
    try:
        label = int(feedback_input.value)
        if label not in [0, 1, 2, 3]:
            raise ValueError
        engine.registrar_feedback(
            _ultimo_resultado['comando'],
            label_correto=label,
            label_predito=_ultimo_resultado['classe_final'],
        )
    except ValueError:
        print('Digite um label válido: 0, 1, 2 ou 3')


def retreinar_interativo(b):
    with saida:
        engine.retreinar_com_feedback()


input_box      = widgets.Text(
    placeholder='Digite o comando (ex: DROP TABLE users)',
    layout=widgets.Layout(width='65%')
)
botao          = widgets.Button(
    description='Avaliar Impacto', button_style='danger',
    layout=widgets.Layout(width='150px')
)
feedback_input = widgets.Text(
    placeholder='Label correto (0–3)',
    layout=widgets.Layout(width='200px')
)
botao_feedback = widgets.Button(
    description='Registrar Feedback', button_style='warning',
    layout=widgets.Layout(width='165px')
)
botao_retreino = widgets.Button(
    description='Re-treinar Modelo', button_style='info',
    layout=widgets.Layout(width='160px')
)
saida = widgets.Output()

botao.on_click(processar_interacao)
botao_feedback.on_click(registrar_feedback_interativo)
botao_retreino.on_click(retreinar_interativo)

display(widgets.VBox([
    widgets.HBox([input_box, botao]),
    widgets.HTML(
        '<br><b>Feedback Loop:</b> '
        'Digite o label correto (0=Baixo, 1=Médio, 2=Alto, 3=Crítico)'
    ),
    widgets.HBox([feedback_input, botao_feedback, botao_retreino]),
    saida,
]))


## 📐 10. Matriz de Pesos Heurística (v5.0)

Documentação completa dos termos heurísticos, seus pesos e o impacto associado.
Termos marcados com ★ são novos na v5.


In [ ]:
NOVOS_V5 = {
    'SUDO KILL -9 1', 'CAT /DEV/NULL > /ETC/SHADOW',
    'TRUNCATE -S 0 /ETC/SHADOW', 'HELM UNINSTALL',
    'AWS RDS DELETE', 'GCLOUD SQL INSTANCES DELETE', 'AWS S3 RB',
}

print(f'  {"Termo Crítico":<40} {"Peso":>6}  {"Impacto"}')
print('  ' + '─' * 75)
for termo, peso in sorted(engine.keywords.items(), key=lambda x: -x[1]):
    if peso >= 90:   impacto = '🔴 Altíssimo (Catastrófico)'
    elif peso >= 60: impacto = '🟠 Alto (Destruição de Estruturas)'
    elif peso >= 30: impacto = '🟡 Médio (Alvo Sensível)'
    else:            impacto = '🟢 Baixo/Médio (Contexto)'
    novo = ' ★' if termo in NOVOS_V5 else ''
    print(f'  {termo:<40} {peso:>6}  {impacto}{novo}')

print('\n  ★ = Novo na v5.0')


## 🏁 11. Conclusão — Trajetória e Próximos Passos

### Trajetória de evolução

O Blast Radius Detector percorreu cinco iterações sucessivas, cada uma atacando
os pontos fracos identificados na versão anterior:

| Versão | Marco principal | F1 weighted |
|---|---|---|
| v1.0 | Prova de conceito — heurística simples | — |
| v2.0 | BART-MNLI + dataset 54 exemplos | 0,479 ± 0,138 |
| v3.0 | XLM-RoBERTa + dataset 180 exemplos | 0,671 ± 0,052 |
| v4.0 | Threshold NLP (50%) + feedback loop + dataset 344 | 0,689 ± 0,032 |
| **v5.0** | **Dataset consistente (502) + re-treino incremental + curva de aprendizado** | **calculado acima** |

### Contribuições técnicas consolidadas

1. **Arquitetura de 3 camadas** com votação ponderada (heurística 50% + RF 25% + NLP 25%)
2. **Threshold de confiança NLP** que previne falsos positivos em comandos inofensivos
3. **Dataset auditável** com assertion de consistência (502 exemplos verificados)
4. **Feedback loop com re-treino incremental** e exportação CSV para pipeline de dados
5. **Curva de aprendizado** para diagnóstico explícito de overfitting
6. **Grafo de propagação dinâmico** com 13 categorias de alvos, incluindo cloud-native

### Próximos passos (v6.0)

- [ ] Dataset ≥ 1000 exemplos com geração assistida por LLM supervisionado
- [ ] Fine-tuning do XLM-RoBERTa em dados de domínio (comandos de infra)
- [ ] Generalização do feedback loop via embeddings (não apenas lookup exato)
- [ ] API REST com rate-limit e autenticação para integração com pipelines de aprovação
- [ ] Testes de carga e latência para SLA em ambientes de produção

---

> **Nota:** Este sistema é um protótipo acadêmico que demonstra a aplicação integrada
> de NLP, ML clássico e heurísticas determinísticas para classificação de risco em
> comandos de infraestrutura. Para implantação em produção, os componentes de segurança,
> autenticação e rate-limiting são requisitos não-negociáveis antes do deploy.
